# LLM and LMM Key Terms

A practitioner brief on the vocabulary that shows up in model cards, API settings, and design reviews: **parameters** and **weights** (what the model *is*), versus **temperature**, **top-p**, **top-k**, and related knobs (how the next token is *chosen*). The same sampling story applies to a **large language model (LLM)** and a **large multimodal model (LMM)** — the extra work in an LMM is turning images (or audio) into tokens before that choice begins.

This notebook is **markdown only**: definitions, architecture, and a worked example. It does not call a model.


## Scope

- What **parameters** and **weights** name, and why a parameter count is a size proxy, not a quality score
- How **tokens**, the **context window**, **logits**, and **softmax** turn a prompt into a next-token distribution
- **Temperature**, **top-p** (nucleus), and **top-k** — what each filters, and when they interact badly
- Related generation controls: **max tokens**, **stop**, **frequency / presence penalty**, **seed**, **logit bias**
- **LLM vs LMM**: same decoder loop; extra input tokens from images (and other media)
- Which knobs OpenAI’s current generation APIs expose, and which belong to other runtimes (notably **top-k**)


## Overview

Two different things get called “parameters” in the same meeting:

| Sense | What it names | Changes when |
|---|---|---|
| **Model parameters** | The learned numbers inside the network (**weights**, plus biases and similar tensors) | Training, fine-tuning, or loading a different checkpoint |
| **Request parameters** | API fields such as `temperature`, `top_p`, `max_output_tokens` | Every call, without touching the checkpoint |

Mixing them up is expensive. Retraining because a reply was “too random” is usually the wrong lever. Turning up surprise on an invoice summary is also the wrong lever.

Generation is **next-token prediction**. The frozen weights score every item in the vocabulary. Sampling knobs reshape that score list, then one token is drawn, decoded, and appended. An LMM runs the same loop after **vision (or audio) tokens** have joined the text tokens in the context window.

OpenAI’s current text surface for new apps is the **Responses API** (`temperature` and `top_p` 0–2 / 0–1, `max_output_tokens` as a hard cap including reasoning tokens). Chat Completions still exists. **Top-k is not an OpenAI request field**; it shows up in Hugging Face, vLLM, Gemini, and similar stacks. Official OpenAI guidance: alter **temperature or top-p, not both**.


## Intuition

**The situation.** A product uses “the model” as if it were one personality. On Monday the reply is a careful summary of an invoice. On Tuesday the same product writes a wild slogan. People argue about whether the system got smarter or broke. Often nothing about the learned system changed — only how freely it was allowed to pick the next word.

**The idea.** Picture a huge library of habits learned from past text (and, for some systems, pictures). That library does not rewrite itself while it answers you. A speaker then reads the next word out loud. When several words would fit, a cautious speaker always takes the most obvious one. A looser speaker sometimes takes a less obvious one, which can sound inventive or sloppy. If the question includes a photo, the speaker is still choosing the next word — they looked at the picture first, then spoke.

**Why it matters.** If you confuse the library with the speaking style, you spend months “retraining” when a dial would have done, or you open the dials on a legal letter and get fluent fiction. Cost tracks how much text and imagery you stuff into the moment of attention. Risk tracks how freely the next word is chosen when the honest answer was “I don’t know.”

**What we are doing here**

- Separate the learned library from the speaking dials
- Name the chunks of text (and picture) the system can hold at once
- Show how the next-word shortlist is narrowed
- Treat a picture-in, text-out path as the same loop with extra input
- Keep facts that must be true in a lookup, not in a lucky next word

**What we are not doing.** A bigger library is not automatically wiser, and a looser speaking style is not more truthful. Surprise is not honesty. A system that can see images is not a different kind of magic — it is the same next-word engine with more kinds of input.


## Architecture

![LLM and LMM next-token inference](images/llm_lmm_key_terms_architecture.png)

**Flow**
1. **Prompt** — the product sends text, and optionally images or other media.
2. **Encode** — a tokenizer (plus an image encoder in an LMM) turns that input into tokens that fit the **context window**.
3. **Forward** — frozen **weights** score the next position (the model does not learn on this request).
4. **Logits** — raw scores over the vocabulary, one number per possible next token.
5. **Sample** — **temperature**, then **top-k** and/or **top-p**, then a draw (or a greedy pick).
6. **Decode** — the chosen id becomes text; it is appended and the loop repeats.
7. **Answer** — generation stops on an end token, a **stop** string, or a **max-token** cap, and the product returns the string.

**Legend:** User Interface (coral) · App plane (magenta) · Data (magenta) · Models (teal) · Orchestration (magenta)

Weights live in **Models**. Sampling knobs live in **Orchestration**. They are not the same object.


## Concepts

### Two vocabularies, one request

```
Checkpoint (does not move at inference)
└── Parameters = weights (+ biases, norms, …)
    └── LLM (text tokens)  or  LMM (text + image/audio tokens)

Request (moves every call)
└── Prompt / messages / images
    └── Context window (token budget)
        └── Logits → temperature → top-k → top-p → draw → decode
            └── Repeat until stop or max tokens
```

### Glossary

| Term | Plain meaning | Production tell |
|---|---|---|
| **LLM** | A large model trained to predict the next **token** of text | Chat, draft, extract, code — text in, text out |
| **LMM** | A large **multimodal** model: more than one input (and sometimes output) type | Screenshot in, caption out; image + question → answer |
| **Parameters** (model) | The learned numbers that implement the network | “8B params”; a size label, not an accuracy SLA |
| **Weights** | The bulk of those numbers (matrices/tensors updated in training) | The checkpoint file you version and roll back |
| **Bias** | A smaller learned offset added to a layer’s output | Part of the parameter count; rarely tuned by itself in product work |
| **Checkpoint** | A saved snapshot of weights (and optimizer state, in training) | `gpt-4o-mini` is a hosted checkpoint family, not a prompt |
| **Inference** | Using frozen weights to produce outputs | Your API call; no gradient, no “the model learned from this chat” |
| **Training / fine-tune** | Updating weights from a dataset | A release, with eval, not a chat turn |
| **Token** | A chunk of text (or an image patch) the model reads or writes | Billing and context are in tokens, not words |
| **Tokenizer** | The map from bytes/text ↔ token ids | Different models tokenize the same sentence differently |
| **Vocabulary** | The finite set of tokens the model can emit | Next-token choice is always over this set |
| **Context window** | Max tokens of input + output the model can consider at once | Prompt + images + completion must fit |
| **Embedding** | A vector for a token or span; nearby ≈ similar | Search, clustering, RAG — not the same as “parameters” |
| **Logits** | Raw, unnormalized scores for each vocabulary item | Before probabilities exist |
| **Softmax** | Turns logits into a probability distribution that sums to 1 | The “pie chart” over next tokens |
| **Greedy decoding** | Always take the highest-probability token | Temperature 0 in practice; repetitive but stable |
| **Temperature** | Divides logits before softmax; higher → flatter distribution | 0 ≈ focused; ~1 default; above 1 more surprising — **not more true** |
| **Top-k** | Keep only the *k* highest-probability tokens, then sample | Integer shortlist; common in open-source servers, **not** OpenAI’s API |
| **Top-p** (nucleus) | Keep the smallest set whose probabilities sum to *p*, then sample | `top_p=0.1` ≈ the top 10% of probability mass |
| **Max tokens** | Hard cap on generated tokens (names vary by API) | Truncation, not “write about this long” |
| **Stop sequence** | Text that ends generation if emitted | Useful for delimited formats |
| **Frequency penalty** | Down-weight tokens in proportion to how often they already appeared | Fights verbatim loops |
| **Presence penalty** | Down-weight tokens that appeared at least once | Pushes toward new topics |
| **Seed** | Best-effort determinism for the same inputs | Not a guarantee; backends drift |
| **Hallucination** | Fluent tokens that are not grounded | Sampling can change *style*; it does not add a truth circuit |
| **Image tokens** | Patches/tiles from a picture that occupy context | LMM cost and window fill; `detail` changes the count |

### How it works (one next token)

1. Pack the **prompt** (and images) into token ids. If they exceed the **context window**, the request fails or is truncated — it is not silently “understood anyway.”
2. Run a **forward pass** through the **weights**. Output: a logit for every vocabulary item.
3. Optional **logit bias** adds a constant to chosen ids.
4. **Temperature** *T*: divide logits by *T* (*T* → 0 sharpens toward the winner; *T* > 1 flattens).
5. **Top-k** (if the runtime has it): zero out everything outside the *k* best.
6. **Top-p**: keep the smallest prefix of remaining mass that reaches *p*.
7. Softmax (on what remains) → sample, or take argmax if greedy.
8. Decode the id to text, append, repeat until **EOS**, **stop**, or **max tokens**.

Typical order in open stacks: temperature scaling → top-k → top-p → sample. OpenAI applies temperature and nucleus sampling; it does not expose *k*.

### When to use which knob

| Job | Prefer | Avoid as the first move |
|---|---|---|
| Extraction, classification, policy text | Temperature **0** (or the lowest the model allows) | High temperature “to sound natural” |
| Drafts that may vary | Temperature ~0.3–0.7 **or** a slightly tightened top-p | Cranking **both** temperature and top-p |
| Brainstorming slogans | Higher temperature **or** higher top-p (one axis) | Treating surprise as a factuality boost |
| Ban a phrase / force a format | Stop sequences, constrained decoding, structured output | Hoping top-k = 5 will “keep it professional” |
| Image question (LMM) | Same sampling policy as the text job; watch **image token** budget | Assuming a photo is “free” in the context window |
| OpenAI-hosted GPT | `temperature` **or** `top_p`; `max_output_tokens` / `max_completion_tokens` | Sending `top_k` as if it were a first-class OpenAI field |
| Reasoning-oriented models | `max_output_tokens` headroom; `reasoning.effort` where documented | Assuming temperature is always a supported lever |

### Comparison

| | Temperature | Top-p | Top-k |
|---|---|---|---|
| What it shapes | Sharpness of the whole distribution | A **mass** cutoff (adaptive shortlist) | A **count** cutoff (fixed shortlist) |
| Typical range | 0–2 (OpenAI) | 0–1 | Integer ≥ 1, or “disabled” |
| Adapts to the step | Yes (flat vs peaked) | Yes (shortlist grows when the model is unsure) | No (always *k* names, even if 3 were obvious) |
| OpenAI Responses / Chat | Yes | Yes | **No** (use other runtimes or `extra_body` only if a compatible server implements it) |
| Official pairing | OpenAI: change this **or** top-p | OpenAI: change this **or** temperature | Independent of OpenAI’s pairing advice |

**Other “k” you will hear (not sampling):** retrieval **top-k** (how many passages to fetch); few-shot **k** (how many examples). Same letter, different object. Name them in the RFC.


## LLM vs LMM

An **LLM** maps a sequence of **text tokens** to the next text token. An **LMM** (also called a vision–language model, VLM, or “multimodal GPT”) accepts **more than one modality** — commonly text + images; some stacks add audio or video. After encoding, both are a **token sequence** plus **weights**. Sampling knobs do not care whether token 17 came from the word “invoice” or from a 512-pixel tile of a screenshot.

| | LLM | LMM |
|---|---|---|
| Input | Text (and tool results as text) | Text **and** images / audio / files the encoder supports |
| Extra cost | Text tokens | **Image tokens** (tiles or patches; `detail` / resolution changes the count) |
| Failure mode | Invented facts in fluent prose | That, plus **missed small print**, wrong coordinates, or “I see X” when the tile was downsized |
| Sampling | Temperature, top-p, (top-k if hosted that way) | **Same knobs** on the text decoder |

OpenAI’s vision path is the same Responses (or Chat Completions) call with image parts. The model “sees” by consuming tokens, not by attaching a separate magic classifier. For fine print and bounding boxes, official guidance is to raise visual **detail** (including `original` where the model supports it) rather than to raise temperature.

**What an LMM is not.** It is not automatically a document database. A photo of a contract still needs retrieval or a system of record if the answer must match the current legal text. It is not a license to skip the context budget: images can dwarf the accompanying question.


## Parameters and weights

In neural nets, **parameters** are the learned scalars. **Weights** are the large matrices that mix signals between layers; **biases** are per-channel offsets. Practitioners say “weights” for the whole checkpoint. A card that reads “70B parameters” means roughly seventy billion learned numbers, not seventy billion facts and not seventy billion “IQ points.”

**Training vs inference**

- **Training** (pre-train, fine-tune, RLHF-style preference tuning) **writes** parameters using a loss and data.
- **Inference** **reads** them. A prompt does not update weights. Few-shot examples in the prompt are **context**, not training.
- **Adapters / LoRA** are small extra parameters you *can* train while leaving the base weights frozen. Still a release, still eval.

**What size buys you.** More parameters often buy more pattern coverage and a higher compute bill. They do not buy a grounded fee schedule, a permission check, or a calibrated fraud score. A small classifier with the right labels can beat a large LLM on a narrow discriminative job.

**Overloaded word.** In an RFC, write **model parameters (weights)** or **sampling / request parameters**. If a sentence still works with either meaning, it is not ready for a design review.


## Tokens, context, and embeddings

**Tokens** are the atomic units of context and billing. English often lands near 3–4 characters per token, but that is a rumor, not a budget. Code, other languages, and punctuation tokenize differently. **Never** estimate image cost as `characters / 4`.

**Context window** = input tokens + output tokens (plus, on reasoning models, **reasoning** and other non-visible tokens that still count against caps). If the window is 128k, a 100k-token prompt leaves 28k for the completion — unless a `max_output_tokens` cap is tighter.

**Embeddings** are vectors used to *retrieve* similar text. They are not the LLM’s weights. Mixing “we stored embeddings” with “we trained the 8B model” is a category error: one is an index, the other is a checkpoint.

**LMM image tokens.** Vision models resize and tile (or patch) the picture. Low detail is cheap and blurry; high / original detail spends more of the window. Official OpenAI token-counting exists so you do not guess. Leave headroom: `max_output_tokens` counts **all** generated tokens, including ones the user never sees.


## Temperature, top-p, and top-k

These knobs do not change what the model *knows*. They change which token is allowed to win **this step**.

**Temperature.** After logits, divide by *T* before softmax. Low *T* makes the winner win harder (OpenAI: lower values such as 0.2 are more focused; 0 is the usual setting for extraction and factual Q&A). High *T* (e.g. 0.8, up to 2) flattens the pie so long-shot tokens get a chance. Temperature is **not** a truthfulness control. OpenAI’s help text is explicit: more random is not more correct.

**Top-p (nucleus sampling).** Sort remaining tokens by probability, walk down until cumulative mass ≥ *p*, discard the tail, then sample. `top_p=0.1` means “only the tokens that make up the top 10% of mass.” When the model is confident, the nucleus may be two tokens; when it is unsure, the nucleus grows. That adaptivity is the point versus a fixed *k*.

**Top-k.** Keep only the *k* highest-probability tokens (e.g. 40 or 50), zero the rest, then sample. Simple and cheap on self-hosted stacks. Rigid: if the next word is obvious, you still keep *k* distractors; if the model is genuinely split among 80 rare names, you cut the list arbitrarily. **OpenAI’s Responses and Chat Completions APIs do not take `top_k`.** LangChain’s `ChatOpenAI` exposes `temperature` and `top_p`; a `top_k` field belongs in `extra_body` only when the **server** is OpenAI-compatible and actually implements it (vLLM, some gateways).

**Do not stack blindly.** Temperature reshapes the pie; top-k and top-p then throw slices away. OpenAI: *“We generally recommend altering this or `top_p` but not both.”* Stacking high temperature with a very high top-p (and a large top-k) is how long completions drift into incoherence.

**Greedy vs sample.** Temperature 0 (argmax) is the production default for anything you will eval as a spec. Sampling is for variety. A `seed` is best-effort determinism, not a contract; OpenAI still tells you to watch backend fingerprints.


## Other generation knobs

| Knob | Role | Watch-out |
|---|---|---|
| **Max tokens** | Hard stop on generation | Does not mean “write *n* tokens.” The model may stop early. Names: `max_tokens` (legacy), `max_completion_tokens` (Chat Completions), `max_output_tokens` (Responses — includes reasoning tokens) |
| **Stop** | End if this string is produced | Easy to clip valid JSON if the stop token appears inside a field |
| **Frequency penalty** | Penalize tokens in proportion to counts so far (−2 to 2 on OpenAI completions-style APIs) | Too high → odd synonyms and broken terms of art |
| **Presence penalty** | Penalize any token already used, encouraging new topics | Can fight a required repeated identifier (account numbers, SKUs) |
| **Logit bias** | Add a constant to specific token ids before sampling | Fragile across tokenizer versions; prefer structured output for schemas |
| **n / best-of** | Several completions, optionally pick one | Multiplies cost; rare in modern chat stacks |
| **Logprobs** | Return log probabilities for debugging | Inspection, not a sampling control |
| **Reasoning effort** | How hard a reasoning model thinks | On some model families, **temperature is fixed or ignored**; do not copy GPT-4o sampling lore onto o-series / GPT-5 reasoning modes without checking the model card |

**Structured output** (`json_schema` / `with_structured_output`) is a *constraint* on the allowed token set. It is a better reliability tool than “temperature 0 and please return JSON” when the contract is a schema.


## How it works end to end

**Offline (the library)**

1. Someone trains or selects a **checkpoint** (LLM or LMM). That is the **parameter** story.
2. You version the **model id**, not “the AI.” Optional: fine-tune or attach adapters — still a release.

**Online (one user request)**

1. The app builds a **prompt** (system + user text; LMM: image parts).
2. **Tokenizer** (+ vision encoder) fills the **context window**.
3. **Forward pass** through frozen **weights** → **logits**.
4. **Sampling policy** (temperature / top-p / top-k / greedy) picks a token.
5. Loop until **stop** or **max tokens**.
6. Product returns text (and logs usage: input, output, reasoning, image tokens).

Changing the system prompt, retrieval, or temperature is **inference-time control**. It is not “the model learned.” If Monday’s replies were stable at temperature 0 and Tuesday’s were not, look at model id, prompt version, and sampling fields before you file a training ticket.


## Walkthrough

Take a request: “Summarize this invoice” with an attached PDF scan.

1. **LMM, not a bigger LLM.** The scan becomes **image tokens**. If `detail` is low, the total might be unreadable; temperature will not restore the missing digits.
2. **Weights** score the next token given the prompt + tiles. They still encode “what invoices usually look like,” which is why they can invent a plausible line item.
3. **Temperature 0** makes the summary repeatable enough to eval. Raising it makes the prose livelier and the amounts less trustworthy.
4. **Top-p** at 1.0 (OpenAI default) means nucleus is off. Tightening to 0.9 is optional; do not also raise temperature.
5. **Top-k** does not appear on the OpenAI call. A self-hosted decoder might set `top_k=40`; that is a server policy, not a GPT-4o switch.
6. **Max output tokens** must cover the summary. Too low → truncated totals. On reasoning models, the same cap also covers hidden thinking tokens.
7. **Stop** is usually unnecessary for a short summary; it matters if you delimited fields with a rare marker.
8. If the total must match the ledger, **retrieve** the structured invoice from the system of record. Sampling cannot be your source of truth.


## Patterns

- **Name both layers in the RFC:** checkpoint (model id, parameter class) and sampling policy (temperature or top-p, max tokens, stop).
- **One sampling axis.** Follow OpenAI: tune temperature **or** top-p. Leave the other at default unless you have an eval that says otherwise.
- **Temperature 0 for contracts.** Extraction, routing, policy restatement, anything with a golden set.
- **Treat LMM cost as tokens.** Budget image `detail` like you budget prompt length.
- **Caps with headroom.** `max_output_tokens` is a ceiling for *all* generated tokens, including reasoning and formatting.
- **Prefer schema constraints** over logit bias and “please output JSON.”
- **Log the knobs.** A quality regression with a silent temperature default change is an incident, not a mystery.
- **Retrieval for facts; sampling for style.** Grounded numbers come from tools and documents, not from a warmer softmax.


## Pitfalls

| Wrong | Right |
|---|---|
| “70B parameters so it won’t hallucinate” | Parameter count is capacity, not a grounding guarantee |
| “The model learned our tone from this chat” | Prompts steer inference; weights change only in training |
| High temperature to make answers “more accurate” | Temperature changes surprise, not truth |
| Tuning temperature **and** top-p together from the first experiment | OpenAI: alter one; A/B the other only with a metric |
| Sending `top_k` to OpenAI and calling it a platform feature | Top-k is a decoder policy on other runtimes |
| `max_tokens=200` to “get a long essay” | It is a cutoff; length is mostly a prompting and stopping problem |
| Same sampling lore on every reasoning model | Some families ignore or forbid temperature; use the model card |
| Image in an LMM with low detail, then blaming the LLM | You starved the encoder of tiles |
| Retrieval **top-k** discussed as if it were sampling **top-k** | Fetch count vs vocabulary shortlist — write both names out |
| Completions-era `max_tokens` as the only name in a new Responses app | Prefer `max_output_tokens`; Chat Completions: `max_completion_tokens` |
| `LLMChain` / `.run()` / `create_react_agent` as “how we set temperature” | Current path: in-cell `ChatOpenAI(..., temperature=..., use_responses_api=True)` and LCEL / `create_agent` / `StateGraph` as the job requires |

**Deprecated or superseded (keep out of new designs):** Completions API as the default for new text apps; mixing Chat Completions `max_tokens` without checking the model’s `max_completion_tokens` migration; `langchain-classic` chains as the way to pass sampling params.


## Variants

| You have | Variant |
|---|---|
| Hosted OpenAI GPT for extraction | Temperature 0, default top-p, schema / structured output |
| Hosted OpenAI GPT for copywriting | Raise temperature **or** tighten/loosen top-p; keep max tokens honest |
| vLLM / Hugging Face / Gemini | You may set **top-k** as well; still pick one primary diversity knob and eval |
| Reasoning model | Budget `max_output_tokens`; tune `reasoning.effort`; do not assume temperature is live |
| Screenshot / PDF page | LMM path; raise visual detail before you raise temperature |
| Must match an internal number | Tool or RAG; sampling policy stays greedy |
| On-prem weights | Same terms; **top-k** may finally exist because you own the sampler |


## Checklist

- [ ] Checkpoint (model id) is named separately from sampling fields
- [ ] “Parameters” in the doc means weights **or** request fields — not both without a qualifier
- [ ] Temperature **or** top-p is the active diversity knob (not an accidental stack)
- [ ] Top-k is only specified if the **runtime** supports it
- [ ] Max-token field matches the API (`max_output_tokens` vs `max_completion_tokens`) and leaves reasoning headroom
- [ ] Stop sequences cannot appear inside legitimate output
- [ ] LMM: image `detail` and token budget reviewed; not estimated as characters/4
- [ ] Retrieval top-k (if any) is documented as a search setting, not a decoder setting
- [ ] Eval set run at the intended temperature (do not eval at 0 and ship at 1)
- [ ] Logs capture model id, temperature, top-p, max tokens, and token usage


## Worked example

**Setting.** A claims ops console has two buttons on the same page: **Draft customer letter** and **Read the attached photo of the odometer**.

**Pass 1 — what is frozen vs what is a dial**

| Piece | Term | Choice |
|---|---|---|
| Hosted vision-capable checkpoint | LMM **weights** / model parameters | One model id for both buttons is fine |
| Letter tone | **Temperature** (or top-p), not a new checkpoint | Letter: 0.4 **or** default temperature with a style prompt; do not also cut top-p to 0.2 on day one |
| Odometer digits | Visual **detail** + temperature **0** | If digits are wrong, raise `detail` / resolution before blaming the LLM |
| “Don’t ramble” | **Max output tokens** + prompt | Cap is a fuse, not a writing teacher |
| “Don’t invent policy” | Retrieval of the current letter template | Sampling will not remember last quarter’s coverage table |

**Pass 2 — the dangerous merge**

Shipping one call with temperature 0.9, top-p 0.95, and a downsized odometer image produces a warm, fluent letter that cites a mileage figure the tiles never supported. The post-incident review says “the 8B vs 70B debate.” The actual miss was **stacked sampling** plus **starved image tokens**. Parameter count never entered the request.

**Pass 3 — success metrics**

- Letter: edit distance vs the sent version; groundedness vs retrieved policy.
- Odometer: exact match on the integer; a separate eval from the letter’s BLEU-ish fluency.
- Cost: input tokens including **image tokens**; output including any reasoning tokens against `max_output_tokens`.

The console still has one model. It has **two sampling policies** and **one visual-token budget**. That is the term-level design.


## Takeaways

- **Weights / model parameters** are the checkpoint. **Temperature, top-p, top-k, penalties, max tokens** are how the next token is chosen. Retrain one; tune the other.
- Generation is a loop: encode → logits → sample → decode → stop. An **LMM** only adds media tokens to the encode step.
- **Temperature** flattens or sharpens the pie. **Top-p** cuts by probability mass. **Top-k** cuts by count. OpenAI exposes the first two and recommends changing **one** of them; top-k lives on other decoders.
- **Max tokens** is a cutoff (Responses: `max_output_tokens`, including hidden tokens). It is not a length instruction.
- Surprise is not truth. Ground numbers with retrieval and tools; use greedy sampling when you will be graded on exactness.
- In later notebooks in this repo, those request fields hang off in-cell `ChatOpenAI` (often `temperature=0`, `use_responses_api=True`). Linear pipes, tool loops, and graphs change *control flow*, not the meaning of a weight or a nucleus.
